# Supplementary source provenance checks

FNCE 2370 — Team 7 | Predicting Corporate Distress Risk

Run this standalone notebook from the project root with the instructor’s raw Parquet file. It exports diagnostic tables only; it does not modify modeling panels or fit models. Outcome-based checks are limited to the initial training window. See `README.md`.

These diagnostics were originally performed before TA clarification. The interpretation below incorporates the construction rules confirmed on November 24, 2026; the checks remain unchanged.

In [1]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

root = Path.cwd()
if root.name == 'notebooks':
    root = root.parent
file_path = root / 'Project_5_data_chs_main_global.parquet'
assert file_path.is_file(), f'File not found: {file_path}'
pf = pq.ParquetFile(file_path)
schema = pf.schema_arrow
names = schema.names
print('Created by:', pf.metadata.created_by)
print('Rows:', pf.metadata.num_rows)
print('Columns:', len(names))

Created by: DuckDB version v1.5.5 (build d8cdaa33fd)
Rows: 3502382
Columns: 223


## 1. Embedded metadata and available fields
Look for provenance, event-type flags, accounting fiscal dates and release timestamps. Generic Parquet writer metadata does not establish research provenance. The pandas and Arrow serialization payloads are not printed in full.

In [2]:
metadata = pf.metadata.metadata or {}
print('File metadata keys:', [k.decode(errors='replace') for k in metadata])
for key, value in metadata.items():
    if key not in [b'pandas', b'ARROW:schema']:
        print(key.decode(errors='replace'), ':', value.decode(errors='replace')[:3000])

print('Schema metadata keys:', list((schema.metadata or {}).keys()))
for field in schema:
    if field.metadata:
        print(field.name, {k.decode(errors='replace'): v.decode(errors='replace')[:1000]
                           for k, v in field.metadata.items()})
patterns = {
    'Failure / event / exit': r'fail|bankrupt|default|rating|exit|delist|dlst|dlret|merg|censor|risk',
    'Accounting availability / release': r'datadate|rdq|fdate|pdate|filing|release|report_date|public_date|avail|fiscal|fyear|fqtr|restat|stale|accounting_date',
    'Merge / source / calendar': r'permno|permco|gvkey|iid|^id$|date|month|match|source'
}
for label, pattern in patterns.items():
    print(f'\n{label}:')
    print([x for x in names if re.search(pattern, x, flags=re.I)])

File metadata keys: []
Schema metadata keys: []

Failure / event / exit:
['first_failure_month', 'first_exit_month', 'first_ambiguous_exit_month', 'at_risk', 'failure_current_month', 'failure_next_month', 'failure_t_plus_1', 'failure', 'failure_within_6m', 'failure_within_12m', 'failure_within_24m', 'failure_within_36m']

Accounting availability / release:
[]

Merge / source / calendar:
['permno', 'date', 'month_end', 'first_failure_month', 'first_exit_month', 'first_ambiguous_exit_month', 'failure_current_month', 'failure_next_month', 'global_factor_matched', 'global_factor_date', 'permco', 'gvkey', 'iid', 'id', 'source_crsp']


## 2. Read a narrow development-period panel
Only initial-training feature months are retained for diagnostics. The supplied first-event dates are retrospective metadata, not predictors. Dates after the cutoff will not be used to count events.

In [3]:
cols = ['permno', 'permco', 'gvkey', 'iid', 'id', 'date', 'month_end',
        'global_factor_eom', 'global_factor_date', 'global_factor_matched',
        'first_failure_month', 'first_exit_month', 'first_ambiguous_exit_month',
        'failure_current_month', 'failure_t_plus_1', 'at_risk', 'excntry',
        'common', 'primary_sec', 'obs_main', 'exch_main', 'source_crsp',
        'me', 'market_equity', 'nimtaavg', 'tlmta', 'exretavg', 'rsize',
        'cashmta', 'mb', 'price_log']
assert set(cols).issubset(names), sorted(set(cols) - set(names))
cutoff = pd.Timestamp('1995-12-31')
# Batch loading avoids materializing the entire raw dataset.
parts = []
for batch in pf.iter_batches(batch_size=100_000, columns=cols):
    part = batch.to_pandas()
    part['month_end'] = pd.to_datetime(part['month_end'], errors='raise')
    parts.append(part.loc[part['month_end'] < cutoff].copy())
d = pd.concat(parts, ignore_index=True)
del parts, part, batch
for col in ['date', 'global_factor_eom', 'global_factor_date', 'first_failure_month',
            'first_exit_month', 'first_ambiguous_exit_month']:
    d[col] = pd.to_datetime(d[col], errors='raise')
d['forecast_month'] = d['month_end'] + pd.offsets.MonthEnd(1)
assert d['forecast_month'].le(cutoff).all()
assert d['month_end'].dt.is_month_end.all()
print('Initial training-window raw rows:', len(d))

Initial training-window raw rows: 1813401


## 3. Original merge: candidate key consistency
The merged file contains only one PERMNO field. Identity of the JKP ID with PERMNO and agreement of calendar dates support a candidate security-month merge, but cannot prove the original SQL/join logic. Duplicate counts below exclude incomplete keys.

In [4]:
m = d.loc[d['global_factor_matched'].eq(1)]
key_sets = [['permno','month_end'], ['id','global_factor_eom'],
            ['permco','global_factor_eom'], ['gvkey','global_factor_eom'],
            ['gvkey','iid','global_factor_eom']]
rows = []
for keys in key_sets:
    ok = m[keys].notna().all(axis=1)
    rows.append({'candidate_key': ' + '.join(keys), 'complete_rows': int(ok.sum()),
                 'missing_key_rows': int((~ok).sum()),
                 'duplicate_excess_rows': int(m.loc[ok, keys].duplicated().sum())})
candidate_keys = pd.DataFrame(rows)
display(candidate_keys)
identifier = pd.to_numeric(m['id'], errors='coerce')
comparable = identifier.notna() & m['permno'].notna()
me1 = pd.to_numeric(m['me'], errors='coerce')
me2 = pd.to_numeric(m['market_equity'], errors='coerce')
me_ok = np.isfinite(me1) & np.isfinite(me2)
merge_evidence = pd.Series({
    'Matched rows': len(m),
    'Comparable numeric JKP ID and PERMNO': int(comparable.sum()),
    'Numeric ID equals PERMNO': int((comparable & identifier.eq(m['permno'])).sum()),
    'Calendar month-end agreement': int(m['month_end'].eq(m['global_factor_eom']).sum()),
    'Exact trading date agreement': int(m['date'].eq(m['global_factor_date']).sum()),
    'Comparable me and market_equity': int(me_ok.sum()),
    'Market equity agreement (within tolerance)': int(np.isclose(me1[me_ok], me2[me_ok], rtol=1e-5, atol=1e-8).sum())
})
display(merge_evidence)

,candidate_key,complete_rows,missing_key_rows,duplicate_excess_rows
0,permno + month_end,1785767,0,0
1,id + global_factor_eom,1785767,0,0
2,permco + global_factor_eom,1785767,0,14442
3,gvkey + global_factor_eom,1594637,191130,9989
4,gvkey + iid + global_factor_eom,1594637,191130,0


Matched rows                                  1785767
Comparable numeric JKP ID and PERMNO          1785767
Numeric ID equals PERMNO                      1785767
Calendar month-end agreement                  1785767
Exact trading date agreement                  1785767
Comparable me and market_equity               1771231
Market equity agreement (within tolerance)    1771231
dtype: int64

## 4. Failure event patterns
Count distinct security/event-month pairs whose event dates occur by the last observed feature month (November 1995). December outcomes are not used for exit/continuation classification because the December observation was not loaded. Compare first-failure dates with recorded exits and observed post-failure trading-panel rows. Neither proximity to an exit nor continuation proves bankruptcy or another particular event type.

In [5]:
event_end = d['month_end'].max()
last_seen = d.groupby('permno')['month_end'].max()
event_records = []
for col in ['first_failure_month','first_exit_month','first_ambiguous_exit_month']:
    known = d.loc[d[col].notna() & d[col].le(event_end), ['permno', col]].drop_duplicates()
    event_records.append({"event": col, "distinct_security_event_months": len(known),
                          "securities_with_multiple_recorded_dates": int(known["permno"].duplicated().sum())})
    print(col, 'distinct security/event-month pairs:', len(known),
          '| securities with multiple recorded dates:', int(known['permno'].duplicated().sum()))

f = d.loc[d['first_failure_month'].notna() & d['first_failure_month'].le(event_end),
          ['permno','first_failure_month','first_exit_month','first_ambiguous_exit_month']].copy()
# Mask event dates outside the observed window before classification.
for col in ['first_exit_month', 'first_ambiguous_exit_month']:
    f[col] = f[col].where(f[col].le(event_end))
f = f.drop_duplicates()
# Do not silently choose a date if retrospective histories disagree.
print('Conflicting event tuples within security/failure month:',
      int(f.duplicated(['permno','first_failure_month']).sum()))
f = f.drop_duplicates(['permno','first_failure_month'], keep=False).copy()
f['last_observed_month'] = f['permno'].map(last_seen)
f['exit_relation'] = np.select([
    f['first_exit_month'].isna(),
    f['first_exit_month'].lt(f['first_failure_month']),
    f['first_exit_month'].eq(f['first_failure_month']),
    f['first_exit_month'].gt(f['first_failure_month'])],
    ['No recorded exit by cutoff','Exit before failure','Exit in failure month','Exit after failure'],
    default='Unclassified')
f['observed_after_failure'] = f['last_observed_month'].gt(f['first_failure_month'])
failure_exit_patterns = pd.crosstab(f['exit_relation'], f['observed_after_failure'], margins=True)
display(failure_exit_patterns)
print('Same-month ambiguous exit and failure:', int(f['first_ambiguous_exit_month'].eq(f['first_failure_month']).sum()))
print('Current failure flags through observed window:', int(d['failure_current_month'].eq(1).sum()))

first_failure_month distinct security/event-month pairs: 2983 | securities with multiple recorded dates: 0
first_exit_month distinct security/event-month pairs: 10138 | securities with multiple recorded dates: 0
first_ambiguous_exit_month distinct security/event-month pairs: 981 | securities with multiple recorded dates: 0
Conflicting event tuples within security/failure month: 0


observed_after_failure,False,All
exit_relation,,
Exit in failure month,2983,2983
All,2983,2983


Same-month ambiguous exit and failure: 0
Current failure flags through observed window: 2983


## 5. Security-month versus company-month
Use only the established initial-training common sample and existing seven-required-feature screen. Compare repeated PERMCO-month and GVKEY-month keys and within-company disagreement in observed targets. As a sensitivity diagnostic, select one security per PERMCO-month using the largest contemporaneous security market equity (`me`), breaking ties with PERMNO. Select before filtering missing labels, so future outcomes cannot determine the representative security. This is not a new baseline specification or a company-level reconstruction of the target.

In [6]:
required = ['nimtaavg','tlmta','exretavg','rsize','cashmta','mb','price_log']
universe = d['global_factor_matched'].eq(1) & d['excntry'].eq('USA') & d['at_risk'].eq(1)
for col in ['common','primary_sec','obs_main','exch_main']:
    universe &= d[col].eq(1)
finite = np.isfinite(d[required].to_numpy(dtype=float)).all(axis=1)
s = d.loc[universe & finite, ['permno','permco','gvkey','forecast_month','me','failure_t_plus_1']].copy()
print('Eligible prediction rows:', len(s))
print('Eligible rows without PERMCO:', int(s['permco'].isna().sum()))
rows = []
for firm_key in ['permco','gvkey']:
    known = s.loc[s[firm_key].notna()]
    g = known.groupby([firm_key,'forecast_month'], sort=False)['failure_t_plus_1']
    counts = g.agg(['size','count','min','max','sum'])
    repeated = counts['size'].gt(1)
    rows.append({'identifier': firm_key, 'rows_with_known_id': len(known),
                 'repeated_company_month_groups': int(repeated.sum()),
                 'excess_security_rows': int((counts['size']-1).sum()),
                 'groups_with_both_0_and_1_labels': int((counts['min'].eq(0)&counts['max'].eq(1)).sum()),
                 'groups_with_multiple_positive_rows': int(counts['sum'].gt(1).sum()),
                 'groups_with_known_and_unknown_labels': int((counts['count'].gt(0)&counts['count'].lt(counts['size'])).sum())})
company_duplicates = pd.DataFrame(rows)
display(company_duplicates)
known = s.loc[s['permco'].notna()].copy()
known['rank_me'] = known['me'].where(np.isfinite(known['me']) & known['me'].gt(0))
representative = known.sort_values(['permco','forecast_month','rank_me','permno'],
                                  ascending=[True,True,False,True], na_position='last').drop_duplicates(['permco','forecast_month'])
rows=[]
for label, frame in [('All eligible securities with PERMCO', known), ('Largest-ME representative per PERMCO-month', representative)]:
    y=frame['failure_t_plus_1']; valid=y.isin([0,1])
    rows.append({'policy':label,'forecast_rows':len(frame),'labeled_rows':int(valid.sum()),
                 'positive_security_outcomes':int(y.eq(1).sum()),'failure_rate_pct':y[valid].mean()*100})
representative_comparison = pd.DataFrame(rows)
display(representative_comparison)

Eligible prediction rows: 1042167
Eligible rows without PERMCO: 0


,identifier,rows_with_known_id,repeated_company_month_groups,excess_security_rows,groups_with_both_0_and_1_labels,groups_with_multiple_positive_rows,groups_with_known_and_unknown_labels
0,permco,1042167,2376,2494,0,2,0
1,gvkey,1041198,2053,2053,0,2,0


,policy,forecast_rows,labeled_rows,positive_security_outcomes,failure_rate_pct
0,All eligible securities with PERMCO,1042167,1042054,2207,0.211793
1,Largest-ME representative per PERMCO-month,1039673,1039560,2205,0.212109


## Interpretation and clarified source conventions

The original execution found unique candidate security-month keys and agreement of all comparable IDs, dates and market equity values. All 2,983 recorded failures through November 1995 coincided with exits. Selecting the largest-market-equity representative retained 99.76% of labeled training observations and 99.91% of positive outcomes. These checks support the current research choices but do not independently reconstruct the upstream join or establish identical company-level model performance.

The course TA clarified on November 24, 2026 that:

- Failure is a narrower CRSP delisting-based measure (codes 550, 552, 560, 561, 574, 580, 584), not the full broad CHS definition. The event-history diagnostics alone cannot establish this classification.
- CHS quarterly accounts become eligible two months after fiscal period-end and carry forward until replaced; six- and twelve-month stale flags are diagnostic, with no age cutoff. Missing weighted predictor inputs and supplied SIGMA_FILL use same-historical-month cross-sectional means.
- Global Characteristics use a four-month accounting lag, an 18-month maximum age from fiscal period-end and item-specific missing-value rules. The merge adds no lag and preserves missingness.
- Direct use of the supplied premerged panel is permitted when no further merging is needed.

These are confirmed construction conventions, not independent checks of individual release dates, exact upstream join code or source-level event codes. The TA acknowledged implementation differences are common; we retain security-months based on our sensitivity analysis and recommend company-level inference. No modeling choices are changed based on test outcomes.

## 6. Export reproducible provenance evidence

Save aggregate diagnostics only. These are descriptive evidence, not model inputs or new sample-selection rules.

In [7]:
tables_dir = root / "outputs" / "tables"
tables_dir.mkdir(parents=True, exist_ok=True)
for name, table in {
    "provenance_candidate_keys": candidate_keys,
    "provenance_merge_evidence": merge_evidence,
    "provenance_event_counts": pd.DataFrame(event_records),
    "provenance_failure_exit_patterns": failure_exit_patterns,
    "provenance_company_duplicates": company_duplicates,
    "provenance_representative_comparison": representative_comparison,
}.items():
    if isinstance(table, pd.Series):
        table.to_csv(tables_dir / f"{name}.csv", header=["value"])
    else:
        table.to_csv(tables_dir / f"{name}.csv", index=name == "provenance_failure_exit_patterns")
print("Provenance diagnostics exported.")

Provenance diagnostics exported.
